# Лабораторна 6. Генерація, доповнена пошуком (RAG)

Власна RAG-система над «Книгою Гравця» D&D 5e (укр.): BM25 + Qwen3-Embedding-0.6B + RRF + bge-reranker-v2-m3 → Qwen3-4B з цитуванням і відмовою.

## 1.1. Корпус

- **Корпус:** «Книга Гравця» настільної рольової гри Dungeons & Dragons 5e, український переклад, «Реліз 1. 2019 р.».
- **Джерело та ліцензія:** Фанатський переклад SRD D&D. PDF у репозиторій не додаю.
- **Формат і обсяг:** 1 PDF, 284 сторінки. Після парсингу в корпусі ≈1,17 млн символів: розділи 1–11 і додатки А–Д. Зміст і алфавітний покажчик виключено.
- **Хто й навіщо питає:** гравці та майстри під час сесії шукають конкретне правило і не хочуть гортати 284 сторінки.
- **Чому RAG, а не промпт чи донавчання:**
  - Корпус на ≈1,17 млн символів, за грубою оцінкою це сотні тисяч токенів, тобто більше за контекст Qwen3-4B. Навіть там, де він вміщується, дорого й повільно на кожне питання.
  - Правила потребують точних цифр і посилання на сторінку для перевірки. Донавчання засвоює факти ненадійно й не дає джерела.
  - RAG дозволяє підміняти редакцію книги без перетренування.

## Особливості корпусу
- Заклинання мають однакову структуру, тому одне заклинання = один фрагмент.
- Таблиці при витягу тексту «розплющуються» в рядок.
- Списки заклинань за класами виключено: багатоколонкова табличка, порядок читання ламається.
- Додаток Г містить параметри істот.

In [ ]:
# [SETUP]
!pip install -q pymupdf langchain-text-splitters

In [ ]:
# [SETUP]
from google.colab import drive
drive.mount("/content/drive")
ARTIFACT_DIR = "/content/drive/MyDrive/ai_systems/lab6"

Mounted at /content/drive


In [ ]:
# [SETUP]
import os, json, datetime
import re
import pymupdf as fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = ARTIFACT_DIR + "/Книга Гравця.pdf"
SPELL_HDR = re.compile(r"^(?:(\d) рівень|Заговір), ([а-яіїєґ'’ʼ\- ]+?)(?: \(([^)]*)\))?$")
SKIP_PAGES = set(range(176, 182))   # списки заклинань за класами: порядок читання ламається

def line_kind(font, size):
    """Тип рядка за шрифтом: h1/h2/h3, 'page' або 'text'."""
    if font.startswith("GoudyOldStyleT"):                return "page"
    if font == "UkrainianGoudyOldBold":                  return "h1"
    if font == "NarrationSRB-Bold" and size >= 17:       return "h2"
    if font == "EngraversGothicBold" and size >= 14:     return "h2"
    if font == "NarrationSRB-Bold" and size >= 11.5:     return "h3"
    if font == "EngraversGothicBold":                    return "h3"
    return "text"

def ordered_blocks(page):
    W = page.rect.width
    blocks = [b for b in page.get_text("dict")["blocks"] if b["type"] == 0]
    blocks.sort(key=lambda b: b["bbox"][1])
    out, band = [], []
    def flush():
        band.sort(key=lambda b: (0 if (b["bbox"][0] + b["bbox"][2]) / 2 < W / 2 else 1, b["bbox"][1]))
        out.extend(band); band.clear()
    for b in blocks:
        if (b["bbox"][2] - b["bbox"][0]) > 0.6 * W:
            flush(); out.append(b)
        else:
            band.append(b)
    flush()
    return out

def extract_lines(doc, first, last, skip_pages=SKIP_PAGES):
    """Рядки в порядку читання. Сторінки нумеруються з 1."""
    res = []
    for pn in range(first - 1, last):
        if (pn + 1) in skip_pages:
            continue
        for bi, b in enumerate(ordered_blocks(doc[pn])):
            for l in b["lines"]:
                t = "".join(s["text"] for s in l["spans"]).strip()
                if not t:
                    continue
                s = l["spans"][0]
                res.append(dict(page=pn + 1, blk=(pn + 1, bi),
                                kind=line_kind(s["font"], s["size"]), text=t, font=s["font"]))
    return res

def join_lines(lines):
    """Склеює рядки в абзац, прибираючи м'які переноси (\\xad)."""
    text = ""
    for t in lines:
        text = text[:-1] + t if text.endswith("\xad") else ((text + " " + t) if text else t)
    return text.replace("\xad", "")

SENT_END = (".", "!", "?", "»", ":", ";", ")", "…")

def parse_sections(lines):
    """Потік рядків -> секції {kind: spell|rules, chapter, section, text, page_start, page_end, ...}."""
    sections, cur = [], None
    chapter, l2 = "", ""
    in_class_chapter = False

    def push_para(p):
        p = p.strip()
        if not p:
            return
        paras = cur["paras"]
        if paras and not paras[-1].startswith("#") and not paras[-1].endswith(SENT_END) and p[0].islower():
            paras[-1] = join_lines([paras[-1], p])        # абзац перетік через колонку/сторінку
        else:
            paras.append(p)

    def flush_buf():
        if cur is not None and cur["buf"]:
            push_para(join_lines(cur["buf"])); cur["buf"] = []

    def close():
        nonlocal cur
        if cur is not None:
            flush_buf()
            cur["text"] = "\n\n".join(cur["paras"]).strip()
            if cur["text"]:
                sections.append(cur)
        cur = None

    def new(kind="rules", page=None, **kw):
        nonlocal cur
        close()
        cur = dict(kind=kind, chapter=chapter, section=l2, paras=[], buf=[], last_blk=None,
                   page_start=page, page_end=page, **kw)

    i, n = 0, len(lines)
    new()
    while i < n:
        ln = lines[i]; k = ln["kind"]
        if k == "page":
            i += 1; continue
        if cur["page_start"] is None:
            cur["page_start"] = ln["page"]
        cur["page_end"] = ln["page"]

        if k in ("h1", "h2", "h3"):
            j, parts = i, []                              # багаторядковий заголовок
            while j < n and lines[j]["kind"] == k:
                parts.append(lines[j]["text"]); j += 1
            head = " ".join(parts)
            nxt = lines[j] if j < n else None
            m = SPELL_HDR.match(nxt["text"]) if (nxt and k == "h3") else None
            if m:                                         # початок опису заклинання
                new("spell", page=ln["page"], name=head, level=int(m.group(1) or 0),
                    school=m.group(2).strip(), ritual=bool(m.group(3)))
                cur["last_blk"] = nxt["blk"]
                i = j + 1; continue
            if k == "h1" and head.startswith(("РОЗДІЛ", "ДОДАТОК")):
                chapter, l2 = head, ""
                in_class_chapter = "КЛАСИ" in head
                new(page=ln["page"])
            elif k == "h1":                               # назва класу великим шрифтом
                l2 = head; new(page=ln["page"])
            elif k == "h2":
                if re.fullmatch(r"―.*―", head):           # декоративні стрічки «― БАРД ―»
                    i = j; continue
                if in_class_chapter and l2:
                    flush_buf(); cur["paras"].append("## " + head)
                else:
                    l2 = head; new(page=ln["page"])
            else:
                flush_buf()
                cur["paras"].append(head if cur["kind"] == "spell" else "### " + head)
            cur["last_blk"] = None
            i = j; continue

        if ln["blk"] != cur["last_blk"]:                  # новий блок PDF = новий абзац
            flush_buf(); cur["last_blk"] = ln["blk"]
        cur["buf"].append(ln["text"])
        i += 1
    close()
    return sections

def slug(s):
    return re.sub(r"\s+", "_", s.strip().lower())

def build_chunks(sections, max_chars=1200, overlap=150, min_chars=80):
    """Заклинання: 1 фрагмент на заклинання. Решта: рекурсивне розбиття.
       title = «розділ › підрозділ» — використовуємо і для ембединга, і для цитування."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chars, chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", "; ", " "], keep_separator="end")
    chunks = []
    for s in sections:
        pages = (s["page_start"], s["page_end"])
        if s["kind"] == "spell":
            head = f'{s["level"]} рівень, {s["school"]}' if s["level"] else f'Заговір, {s["school"]}'
            if s["ritual"]:
                head += " (ритуал)"
            chunks.append(dict(doc_id="spell:" + slug(s["name"]), kind="spell",
                               title=f'Заклинання: {s["name"]}', text=f'{head}\n{s["text"]}',
                               page_start=pages[0], page_end=pages[1],
                               level=s["level"], school=s["school"]))
            continue
        if len(s["text"]) < min_chars:
            continue
        title = " › ".join(x for x in (s["chapter"], s["section"]) if x) or "Вступ"
        for part in splitter.split_text(s["text"]):
            if len(part) >= min_chars:
                chunks.append(dict(doc_id="rules:" + slug(title), kind="rules", title=title,
                                   text=part.strip(), page_start=pages[0], page_end=pages[1]))
    for i, c in enumerate(chunks):
        c["chunk_id"] = i
    return chunks

CONFIGS = {"600/100": (600, 100), "1200/150": (1200, 150), "2000/250": (2000, 250)}

In [ ]:
import collections, statistics

doc = fitz.open(PDF_PATH)
sections = parse_sections(extract_lines(doc, 9, 279))   # с. 1–8 зміст, 280+ покажчик

CONFIGS = {"600/100": (600, 100), "1200/150": (1200, 150), "2000/250": (2000, 250)}
CHUNKS = {name: build_chunks(sections, size, ov) for name, (size, ov) in CONFIGS.items()}

for name, ch in CHUNKS.items():
    r = [len(c["text"]) for c in ch if c["kind"] == "rules"]
    print(f"{name:9} фрагментів: {len(ch):5}  {dict(collections.Counter(c['kind'] for c in ch))}  "
          f"rules: медіана {statistics.median(r):.0f}, макс {max(r)} симв.")

# приклад фрагмента-заклинання
c = next(c for c in CHUNKS["1200/150"] if c["title"] == "Заклинання: ВОГНЯНА КУЛЯ")
print(c["title"], "| с.", c["page_start"], "\n", c["text"][:500])

600/100   фрагментів:  2195  {'rules': 1834, 'spell': 361}  rules: медіана 479, макс 599 симв.
1200/150  фрагментів:  1200  {'rules': 839, 'spell': 361}  rules: медіана 1030, макс 1199 симв.
2000/250  фрагментів:   868  {'rules': 507, 'spell': 361}  rules: медіана 1811, макс 1998 симв.
Заклинання: ВОГНЯНА КУЛЯ | с. 191 
 3 рівень, втілення
Час виконання: 1 вчинок Дистанція: 30 метрів Компоненти: С, Т, М (крихітна кулька з гуано кажана і сірки) Тривалість: Миттєва Яскравий промінь вилітає із твого вказівного пальця в точку, обрану тобою в межах дистанції, де і відбувається вибух полум’я з гучним гуркотом. Всі істоти в межах сфери з радіусом 4 метри з центром в тій точці повинні зробити кидок рятунку Вправності. Ціль отримує 8к6 ушкоджень вогнем при провалі кидка рятунку, або половину цих ушкоджень при успіху. Цей


## 1.2. Бенчмарк

26 питань складено **до** налаштування системи. Для кожного вказано еталонну відповідь, документ із відповіддю і вираз-доказ. Фрагмент вважається правильним, якщо він із правильного документа **і** містить вираз-доказ, як у лекції.

| Тип | К-сть | Приклад |
|---|---|---|
| точні факти про заклинання | 6 | дистанція, шкода, компоненти «Вогняної кулі» |
| перефразування без назви | 2 | «заклинання, що виганяє в лабіринт» |
| загальні правила | 6 | відпочинок, критичне попадання, стан «осліплений» |
| раса / клас | 4 | швидкість дверга, кубик здоров'я чарівника |
| спорядження | 2 | ціна довгого меча, КО кольчуги |
| таблиця | 1 | досвід для 5 рівня |
| істота | 1 | здоров'я вовка |
| **без відповіді в корпусі** | 4 | беголдер, магічний меч, дракон, фланг |

Відсутність відповіді на 4 останні питання перевірено пошуком по тексту. «Небезпечність» у книзі є лише в додатку Г для інших істот, тож питання про беголдера перевіряє ще й стійкість до схожих, але хибних фрагментів.

In [ ]:
# [SETUP]
# (id, тип, питання, [doc_id з відповіддю], regex-доказ, еталонна відповідь)
BENCH = [
 ("q01","заклинання","Яка дистанція дії заклинання «Вогняна куля»?",["spell:вогняна_куля"],r"Дистанція: 30 метрів","30 метрів."),
 ("q02","заклинання","Скільки ушкоджень вогнем завдає «Вогняна куля» при провалі кидка рятунку?",["spell:вогняна_куля"],r"8к6 ушкоджень вогнем","8к6 ушкоджень вогнем (половина при успіху)."),
 ("q03","заклинання","Скільки триває заклинання «Левітація» і чи потрібна концентрація?",["spell:левітація"],r"Концентрація, не довше 10 хвилин","Так, концентрація, не довше 10 хвилин."),
 ("q04","заклинання","Який рівень має заклинання «Бажання» і до якої школи магії належить?",["spell:бажання"],r"9 рівень, виклик","9 рівень, виклик."),
 ("q05","заклинання","Які матеріальні компоненти потрібні для «Ланцюгової блискавки»?",["spell:ланцюгова_блискавка"],r"шматочок хутра","Шматочок хутра; шматок бурштину, скла або кристалічний жезл; три срібні шпильки."),
 ("q06","заклинання","Яку шкоду завдає «Ланцюгова блискавка» і на скільки додаткових цілей вона перестрибує?",["spell:ланцюгова_блискавка"],r"10к8","10к8 ушкоджень електрикою; перестрибує на три додаткові цілі."),
 ("q07","парафраз","Яке заклинання виганяє істоту в деміплощину, схожу на лабіринт?",["spell:лабіринт"],r"деміплощину, схожу на лабіринт","«Лабіринт» (8 рівень, виклик)."),
 ("q08","парафраз","Яке заклинання дозволяє підняти істоту вертикально вгору приблизно на 4 метри?",["spell:левітація"],r"4 метрів","«Левітація»."),
 ("q09","правила","Скільки триває тривалий відпочинок?",["rules:розділ_8._пригоди_›_відпочинок"],r"щонайменше 8 годин","Щонайменше 8 годин."),
 ("q10","правила","Скільки триває короткий відпочинок?",["rules:розділ_8._пригоди_›_відпочинок"],r"мінімум в 1 годину","Мінімум 1 година."),
 ("q11","правила","Що відбувається, якщо при кидку рятунку від смерті випадає «1»?",["rules:розділ_9._бій_›_ушкодження_та_лікування"],r"випадає «1»[^.]*двома провалами","Це вважається двома провалами."),
 ("q12","правила","Як рахується вантажопідйомність персонажа?",["rules:розділ_7._використання_характеристик_›_використання_характеристик"],r"показник Сили, помножений на 7","Показник Сили × 7 (у кілограмах)."),
 ("q13","правила","Що відбувається з кубиками ушкоджень при критичному попаданні?",["rules:розділ_9._бій_›_ушкодження_та_лікування"],r"кубики ушкоджень по цілі два рази","Кубики ушкоджень кидаються двічі, потім додаються модифікатори."),
 ("q14","правила","Що відбувається з осліпленою істотою під час атак?",["rules:додаток_а._стани"],r"Кидки атак по такій істоті відбуваються з перевагою","Вона автоматично провалює перевірки, пов'язані із зором; атаки по ній — з перевагою, її атаки — з перешкодою."),
 ("q15","раса","Яка базова швидкість ходьби дверга?",["rules:розділ_2._раси_›_дверг"],r"базова швидкість ходьби становить 5 метрів","5 метрів."),
 ("q16","раса","Якими мовами розмовляє дверг?",["rules:розділ_2._раси_›_дверг"],r"загальній та двергській","Загальною та двергською."),
 ("q17","клас","Який кубик здоров'я має чарівник?",["rules:розділ_3._класи_›_чарівник"],r"1к6 за кожен рівень чарівника","1к6 за кожен рівень."),
 ("q18","клас","Що дає варвару стан люті і чим її запускають?",["rules:розділ_3._класи_›_варвар"],r"бонусний вчинок, щоб у свій хід увійти в стан люті","Лють вмикається бонусним вчинком; перевага на перевірки й кидки рятунку Сили тощо."),
 ("q19","спорядження","Скільки коштує довгий меч і яку шкоду він завдає?",["rules:розділ_5._спорядження_›_зброя"],r"Довгий меч 15 зм 1к8 рубаних","15 зм, 1к8 рубаних ушкоджень."),
 ("q20","спорядження","Який Клас Обладунку дає кольчуга і яка її ціна?",["rules:розділ_5._спорядження_›_обладунки_та_щити"],r"Кольчуга 75 зм 16","75 зм, КО 16."),
 ("q21","таблиця","Скільки очок досвіду потрібно для 5 рівня?",["rules:розділ_1._поетапне_створення_персонажа_›_після_1_рівня"],r"6,500 5 \+3","6 500 досвіду."),
 ("q22","істота","Скільки здоров'я має вовк і який у нього Клас Обладунку?",["rules:додаток_г._параметри_істот_›_вовк"],r"Клас Обладунку 13[^.]*Здоров’я 11","КО 13, здоров'я 11 (2к8 + 2)."),
 ("q23","без відповіді","Який показник небезпечності має беголдер?",[],None,None),
 ("q24","без відповіді","Скільки коштує магічний меч +1 і як його створити?",[],None,None),
 ("q25","без відповіді","Які легендарні дії має дорослий червоний дракон?",[],None,None),
 ("q26","без відповіді","Як працює правило флангу і яку перевагу воно дає?",[],None,None),
]
print(len(BENCH), "питань;", sum(1 for q in BENCH if not q[3]), "без відповіді")

26 питань; 4 без відповіді


In [ ]:
def gold_chunks(q, chunks):
    _, _, _, docs, ev, _ = q
    return [c["chunk_id"] for c in chunks if c["doc_id"] in docs and re.search(ev, c["text"])]

for name, ch in CHUNKS.items():
    missing = [q[0] for q in BENCH if q[3] and not gold_chunks(q, ch)]
    print(name, "— питань без золотого фрагмента:", missing or "немає")

600/100 — питань без золотого фрагмента: немає
1200/150 — питань без золотого фрагмента: немає
2000/250 — питань без золотого фрагмента: немає


## 1.3. Індексація

| Рішення | Вибір | Обґрунтування |
|---|---|---|
| Модель ембедингів | `Qwen/Qwen3-Embedding-0.6B` | багатомовна, добре працює з українською, вміщується в Colab T4; запит кодується з інструкцією |
| Розбиття | заклинання - 1 фрагмент, решта книги — розбиття | опис заклинання змістовно завершений; для решти розмір порівнюємо експериментально |
| Розмір / перекриття | 600/100, 1200/150, 2000/250 символів | три конфігурації для порівняння в 1.4 |
| Контекст фрагмента | на початок додається «Розділ › підрозділ» | фрагмент про «швидкість 5 метрів» без слова «дверг» втрачає зміст |
| Векторне сховище | FAISS `IndexFlatIP` по нормалізованих векторах | для ≈1–2 тис. векторів точний пошук миттєвий |
| Лексичний пошук | BM25 `rank_bm25` з лематизацією `pymorphy3` | в українській багато відмінкових форм; без лематизації BM25 їх не зв'язує |
| `max_seq_length` моделі | 1024 токени | щоб фрагменти до 2000 символів не обрізались |

## 1.4. Ретрив

Метрики рахуються по 22 питаннях із відповіддю. Правильний фрагмент має бути з правильного документа **і** містити вираз-доказ. Чотири питання без відповіді в ретриві не оцінюються, їх використаємо в 1.5 для відмов.

In [ ]:
# [SETUP]
!pip install -q rank_bm25 pymorphy3 pymorphy3-dicts-uk

In [ ]:
# [SETUP]
import numpy as np
import pymorphy3
from functools import lru_cache
from rank_bm25 import BM25Okapi

morph = pymorphy3.MorphAnalyzer(lang="uk")

@lru_cache(maxsize=None)
def lemma(w):
    return morph.parse(w)[0].normal_form

TOKEN = re.compile(r"[а-яіїєґa-z0-9]+(?:[’'\-][а-яіїєґa-z0-9]+)*", re.I)

def tokenize(text):
    return [lemma(t.lower()) for t in TOKEN.findall(text)]

def doc_text(c):
    """Текст для індексації: назва розділу + фрагмент."""
    return f'{c["title"]}\n{c["text"]}'

class BM25Index:
    def __init__(self, chunks):
        self.bm25 = BM25Okapi([tokenize(doc_text(c)) for c in chunks])
    def search(self, query, k=10):
        scores = self.bm25.get_scores(tokenize(query))
        return list(np.argsort(-scores)[:k])

def rrf(rankings, k_rrf=60, top=10):
    """Reciprocal Rank Fusion: сума 1/(k_rrf + позиція), позиції з 1."""
    score = {}
    for ranking in rankings:
        for pos, cid in enumerate(ranking, start=1):
            score[cid] = score.get(cid, 0.0) + 1.0 / (k_rrf + pos)
    return [cid for cid, _ in sorted(score.items(), key=lambda x: -x[1])[:top]]

def is_gold(q, c):
    _, _, _, docs, ev, _ = q
    return c["doc_id"] in docs and re.search(ev, c["text"]) is not None

def evaluate(search_fn, chunks, bench, ks=(1, 3, 5, 10)):
    """search_fn(query, k) -> список chunk_id. Питання без відповіді не враховуються."""
    qs = [q for q in bench if q[3]]
    ranks = []
    for q in qs:
        ids = search_fn(q[2], max(ks))
        ranks.append(next((i + 1 for i, cid in enumerate(ids) if is_gold(q, chunks[cid])), None))
    res = {f"hit@{k}": float(np.mean([r is not None and r <= k for r in ranks])) for k in ks}
    res["MRR@10"] = float(np.mean([1 / r if r else 0.0 for r in ranks]))
    return res, dict(zip([q[0] for q in qs], ranks))

In [ ]:
import pandas as pd

RANKS = {}
rows = []
for name, ch in CHUNKS.items():
    bm = BM25Index(ch)
    res, ranks = evaluate(lambda q, k: bm.search(q, k), ch, BENCH)
    RANKS[(name, "BM25")] = ranks
    rows.append({"розбиття": name, "метод": "BM25", **res})
pd.DataFrame(rows).round(2)

,розбиття,метод,hit@1,hit@3,hit@5,hit@10,MRR@10
0,600/100,BM25,0.45,0.73,0.73,0.77,0.58
1,1200/150,BM25,0.55,0.73,0.77,0.86,0.66
2,2000/250,BM25,0.55,0.77,0.82,0.86,0.67


In [ ]:
# [SETUP]
!pip install -q sentence-transformers faiss-cpu

In [ ]:
# [SETUP]
import torch, faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
emb_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
emb_model.max_seq_length = 1024
print("розмірність:", emb_model.get_sentence_embedding_dimension(), "| max_seq_length:", emb_model.max_seq_length)

class DenseIndex:
    def __init__(self, chunks):
        emb = emb_model.encode([doc_text(c) for c in chunks], batch_size=16,
                               normalize_embeddings=True, show_progress_bar=True,
                               convert_to_numpy=True).astype("float32")
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb)
    def search_scored(self, query, k=10):
        q = emb_model.encode([query], prompt_name="query", normalize_embeddings=True,
                             convert_to_numpy=True).astype("float32")
        scores, ids = self.index.search(q, k)
        return list(ids[0]), list(scores[0])
    def search(self, query, k=10):
        return self.search_scored(query, k)[0]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

розмірність: 1024 | max_seq_length: 1024


/tmp/ipykernel_490/2060259651.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("розмірність:", emb_model.get_sentence_embedding_dimension(), "| max_seq_length:", emb_model.max_seq_length)


In [ ]:
# [SETUP]

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, max_length=1024)

def hybrid(bm, dn, q, k, pool=20):
    return rrf([bm.search(q, pool), dn.search(q, pool)], top=k)

def hybrid_rerank(bm, dn, chunks, q, k, pool=20):
    cand = hybrid(bm, dn, q, pool, pool)
    scores = reranker.predict([(q, doc_text(chunks[i])) for i in cand])
    return [cand[i] for i in np.argsort(-scores)[:k]]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [ ]:
# Порівняння 3 розмірів фрагмента × 4 методів пошуку. Кодує корпус тричі, тому за замовчуванням вимкнено:
# результати запуску наведено у виводі нижче та у звіті 1.8. Щоб відтворити - RUN_GRID = True.
import pandas as pd
RUN_GRID = False
GRID_FILE = f"{ARTIFACT_DIR}/grid_results.json"

if RUN_GRID:
    INDEX, rows = {}, []
    for name, ch in CHUNKS.items():
        bm, dn = BM25Index(ch), DenseIndex(ch)
        INDEX[name] = (bm, dn)
        methods = {
            "BM25":            lambda q, k: bm.search(q, k),
            "семантичний":     lambda q, k: dn.search(q, k),
            "гібрид (RRF)":    lambda q, k: hybrid(bm, dn, q, k),
            "гібрид + rerank": lambda q, k: hybrid_rerank(bm, dn, ch, q, k),
        }
        for m, fn in methods.items():
            res, ranks = evaluate(fn, ch, BENCH)
            rows.append({"розбиття": name, "метод": m, **res})
    df = pd.DataFrame(rows).round(2)
    df.to_json(GRID_FILE, force_ascii=False, orient="records")
elif os.path.exists(GRID_FILE):
    df = pd.read_json(GRID_FILE)
else:
    df = None
    print("RUN_GRID = False і збереженого файлу немає: таблиця — у виводі цієї клітинки та у звіті 1.8")
df

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

... (4 outputs left out)

Batches:   0%|          | 0/138 [00:00<?, ?it/s]

Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Batches:   0%|          | 0/55 [00:00<?, ?it/s]

,розбиття,метод,hit@1,hit@3,hit@5,hit@10,MRR@10
0,600/100,BM25,0.45,0.73,0.73,0.77,0.58
1,600/100,семантичний,0.64,0.77,0.82,0.86,0.71
2,600/100,гібрид (RRF),0.68,0.73,0.82,0.91,0.73
3,600/100,гібрид + rerank,0.77,0.91,0.95,0.95,0.84
4,1200/150,BM25,0.55,0.73,0.77,0.86,0.66
5,1200/150,семантичний,0.59,0.82,0.86,0.91,0.70
6,1200/150,гібрид (RRF),0.68,0.82,0.91,0.95,0.78
7,1200/150,гібрид + rerank,0.82,0.91,0.95,0.95,0.86
8,2000/250,BM25,0.55,0.77,0.82,0.86,0.67
9,2000/250,семантичний,0.77,0.86,0.91,0.95,0.83


### Висновки з порівняння конфігурацій

- Семантичний пошук кращий за BM25 для всіх трьох розмірів фрагмента.
- Переранжування дає найбільший приріст саме на hit@1: гібрид → гібрид + rerank.
- Без переранжування великі фрагменти допомагають семантичному пошуку. З переранжуванням різниця майже зникає: hit@1 0.77 / 0.82 / 0.82.
- Одне питання бенчмарка — це 4,5 п.п. Різниця MRR між `1200/150` - 0.86 і `2000/250` - 0.88 менша за одне питання, тож розрізнити ці конфігурації 22 питаннями не можна.
- Для подальшої роботи обрано **`1200/150` + гібрид + rerank**: метрики практично збігаються з `2000/250`, а фрагменти вдвічі коротші, тому контекст для генератора дешевший і джерело легше перевірити.

In [ ]:
# [SETUP]

def _norm(s):
    return re.sub(r"[\s\xad]+", "", s)

def refine_pages(chunks, doc, probe=40):
    cache = {}
    def ptxt(p):
        if p not in cache:
            cache[p] = _norm(doc[p - 1].get_text())
        return cache[p]
    fixed = 0
    for c in chunks:
        if c.get("refined"):
            continue
        c["refined"] = True
        body = _norm(re.sub(r"^#+ ", "", c["text"], flags=re.M))   # без markdown-маркерів заголовків
        head, tail = body[:probe], body[-probe:]
        lo, hi = c["page_start"], min(c["page_end"], len(doc))
        ps = next((p for p in range(lo, hi + 1) if head in ptxt(p)), None)
        if ps is None:
            continue
        pe = next((p for p in range(ps, hi + 1) if tail in ptxt(p)), min(hi, ps + 1))
        fixed += (ps, pe) != (lo, hi)
        c["page_start"], c["page_end"] = ps, max(ps, pe)
    return fixed

BEST = "1200/150"   # конфігурація, обрана за результатами 1.4
K = 5               # скільки фрагментів передаємо генератору

def _paths(best):
    tag = best.replace("/", "_")
    return (f"{ARTIFACT_DIR}/dense_{tag}.faiss", f"{ARTIFACT_DIR}/chunks_{tag}.json",
            f"{ARTIFACT_DIR}/meta_{tag}.json")

def _dump_chunks(path, chunks):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)

def save_artifacts(best, chunks, dn):
    p_idx, p_ch, p_meta = _paths(best)
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    faiss.write_index(dn.index, p_idx)
    _dump_chunks(p_ch, chunks)
    with open(p_meta, "w", encoding="utf-8") as f:
        json.dump(dict(chunking=best, n_chunks=len(chunks), pages_refined=True,
                       embedding_model="Qwen/Qwen3-Embedding-0.6B", max_seq_length=1024,
                       indexed_text="title + '\\n' + text", reranker="BAAI/bge-reranker-v2-m3",
                       saved=str(datetime.date.today())), f, ensure_ascii=False, indent=1)

def load_or_build(best=BEST):
    p_idx, p_ch, _ = _paths(best)
    if os.path.exists(p_idx) and os.path.exists(p_ch):
        with open(p_ch, encoding="utf-8") as f:
            chunks = json.load(f)
        dn = DenseIndex.__new__(DenseIndex)            # без перекодування корпусу
        dn.index = faiss.read_index(p_idx)
        assert dn.index.ntotal == len(chunks), "індекс і фрагменти не збігаються"
        print("завантажено з диска:", len(chunks), "фрагментів")
        if not all(c.get("refined") for c in chunks):
            if os.path.exists(PDF_PATH):
                n = refine_pages(chunks, fitz.open(PDF_PATH))
                _dump_chunks(p_ch, chunks)
                print("сторінки уточнено для", n, "фрагментів і збережено")
            else:
                print("PDF не знайдено: сторінки лишилися діапазонами секцій")
    else:
        if not os.path.exists(PDF_PATH):
            raise FileNotFoundError(f"Немає індексу в {ARTIFACT_DIR} і немає PDF: {PDF_PATH}")
        doc = fitz.open(PDF_PATH)
        size, ov = CONFIGS[best]
        chunks = build_chunks(parse_sections(extract_lines(doc, 9, 279)), size, ov)
        refine_pages(chunks, doc)
        dn = DenseIndex(chunks)
        save_artifacts(best, chunks, dn)
        print("зібрано з PDF і збережено:", len(chunks), "фрагментів")
    return chunks, BM25Index(chunks), dn

CH, BM, DN = load_or_build(BEST)

завантажено з диска: 1200 фрагментів
сторінки уточнено для 790 фрагментів і збережено


In [ ]:
import torch
print(torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"вільно {free/1e9:.1f} ГБ з {total/1e9:.1f} ГБ")

Tesla T4
вільно 12.0 ГБ з 15.6 ГБ


In [ ]:
# [SETUP]
emb_model.half()
reranker.model.half()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"після fp16: вільно {free/1e9:.1f} ГБ")

після fp16: вільно 12.9 ГБ


## 1.5. Генерація з цитуванням

| Рішення | Вибір |
|---|---|
| Генеративна модель | Qwen 4B, fp16, локально в ноутбуці через `transformers` |
| Режим | `do_sample=False`, без режиму міркування |
| Контекст | `k = 5` фрагментів після гібридного пошуку з переранжуванням |
| Цитування | фрагменти нумеруються `[1]…[5]` разом із розділом і сторінкою |
| Відмова | якщо у фрагментах немає відповіді, модель має відповісти фіксованою фразою |

**Метрики на бенчмарку:**
- *правильність* — вердикт LLM-судді (та сама Qwen 4B) порівняно з еталоном; відмова на питання з відповіддю вважається неправильною;
- *золотий фрагмент у контексті* — чи потрапила відповідь у `k` фрагментів;
- *правильна і цитує золотий фрагмент* — коректність цитування: серед процитованих `[n]` є фрагмент з правильного документа з виразом-доказом;
- *коректна відмова* — на 4 питаннях без відповіді система відмовилась і не поставила посилань.

In [ ]:
# [SETUP]

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"   # якщо в тебе звичайна Qwen/Qwen3-4B — заміни тут
tok = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
llm.eval()

free, total = torch.cuda.mem_get_info()
print(f"вільно після завантаження: {free/1e9:.1f} ГБ з {total/1e9:.1f} ГБ")

def chat(messages, max_new_tokens=300):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)       # для Qwen3-4B вимикає режим міркування
    inputs = tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    n_in = inputs["input_ids"].shape[1]
    return tok.decode(out[0][n_in:], skip_special_tokens=True).strip(), n_in

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

вільно після завантаження: 5.1 ГБ з 15.6 ГБ


In [ ]:
# [SETUP]

SYSTEM = """Ти — помічник з правил D&D 5e. Відповідай українською і лише на основі наданих фрагментів «Книги Гравця».
Правила:
1. Після кожного твердження став посилання на джерело у вигляді [n], де n — номер фрагмента.
2. Не використовуй знання поза фрагментами і нічого не вигадуй.
3. Якщо у фрагментах немає відповіді на питання, відповідай рівно так: «У наданих фрагментах немає відповіді на це питання.» — без посилань і без здогадів.
4. Відповідь — 1–4 речення з точними числами й назвами з тексту."""

def pages(c):
    return f"с. {c['page_start']}"      # сторінка, де фрагмент починається

def build_messages(question, ctx):
    blocks = [f"[{i}] ({c['title']}, {pages(c)})\n{c['text']}" for i, c in enumerate(ctx, start=1)]
    user = "Фрагменти:\n\n" + "\n\n".join(blocks) + f"\n\nПитання: {question}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def is_refusal(ans):
    return "немає відповіді" in ans.lower()

def cited_numbers(ans, n_ctx):
    nums = set()
    for grp in re.findall(r"\[(\d+(?:\s*,\s*\d+)*)\]", ans):
        nums.update(int(x) for x in re.split(r"\s*,\s*", grp))
    return sorted(n for n in nums if 1 <= n <= n_ctx)

JUDGE_SYS = ("Ти суддя. Порівняй відповідь системи з еталоном. Відповідай одним словом: "
             "ВІРНО — якщо відповідь містить суть еталона (усі ключові числа й факти) без суперечностей; "
             "інакше НЕВІРНО.")

def judge_messages(question, reference, answer):
    return [{"role": "system", "content": JUDGE_SYS},
            {"role": "user", "content": f"Питання: {question}\nЕталон: {reference}\nВідповідь системи: {answer}\nВердикт:"}]

def verdict_ok(v):
    return v.strip().upper().startswith("ВІРНО")     # «НЕВІРНО» не починається з «ВІРНО»

def retrieve(question, k=K):
    return [int(i) for i in hybrid_rerank(BM, DN, CH, question, k)]

def answer(question, k=K):
    ids = retrieve(question, k)
    text, n_tok = chat(build_messages(question, [CH[i] for i in ids]))
    return dict(question=question, answer=text, ctx_ids=ids, prompt_tokens=n_tok)

def run_benchmark(bench, k=K):
    rows = []
    for q in bench:
        qid, typ, question, docs, ev, ref = q
        r = answer(question, k)
        cited = [r["ctx_ids"][n - 1] for n in cited_numbers(r["answer"], len(r["ctx_ids"]))]
        row = dict(id=qid, type=typ, answerable=bool(docs), answer=r["answer"], ctx_ids=r["ctx_ids"],
                   refused=is_refusal(r["answer"]), n_cited=len(cited), prompt_tokens=r["prompt_tokens"])
        if docs:
            row["gold_in_context"] = any(is_gold(q, CH[i]) for i in r["ctx_ids"])
            row["cites_gold"] = any(is_gold(q, CH[i]) for i in cited)
            v, _ = chat(judge_messages(question, ref, r["answer"]), max_new_tokens=8)
            row["judge"] = verdict_ok(v) and not row["refused"]
        rows.append(row)
    return rows

def summarize(rows):
    A = [r for r in rows if r["answerable"]]; N = [r for r in rows if not r["answerable"]]
    m = lambda xs: sum(xs) / len(xs)
    return {
        "відповідь правильна (суддя), питання з відповіддю": m([r["judge"] for r in A]),
        "золотий фрагмент у контексті (hit@k)":             m([r["gold_in_context"] for r in A]),
        "правильна І цитує золотий фрагмент":               m([r["judge"] and r["cites_gold"] for r in A]),
        "є хоч одне посилання [n]":                         m([r["n_cited"] > 0 for r in A]),
        "хибна відмова (відповідь у корпусі є)":            m([r["refused"] for r in A]),
        "коректна відмова, питання без відповіді":          m([r["refused"] and r["n_cited"] == 0 for r in N]),
    }

In [ ]:
import pandas as pd
ROWS = run_benchmark(BENCH, k=K)
with open(f"{ARTIFACT_DIR}/rag_rows.json", "w", encoding="utf-8") as f:
    json.dump(ROWS, f, ensure_ascii=False, indent=1)
print(pd.Series(summarize(ROWS)).round(2).to_string())
pd.DataFrame(ROWS)[["id", "type", "refused", "n_cited", "gold_in_context", "cites_gold", "judge", "answer"]]

відповідь правильна (суддя), питання з відповіддю    0.77
золотий фрагмент у контексті (hit@k)                 0.95
правильна І цитує золотий фрагмент                   0.77
є хоч одне посилання [n]                             0.95
хибна відмова (відповідь у корпусі є)                0.09
коректна відмова, питання без відповіді              1.00


,id,type,refused,n_cited,gold_in_context,cites_gold,judge,answer
0,q01,заклинання,False,1,True,True,True,Дистанція дії заклинання «Вогняна куля» — 30 м...
1,q02,заклинання,False,1,True,True,True,«Вогняна куля» при провалі кидка рятунку завда...
2,q03,заклинання,False,1,True,True,True,Заклинання «Левітація» триває не довше 10 хвил...
3,q04,заклинання,True,4,True,True,False,Заклинання «Бажання» має 9 рівень [1]. У надан...
4,q05,заклинання,False,1,True,True,True,Для заклинання «Ланцюгової блискавки» потрібні...
5,q06,заклинання,False,1,True,True,False,«Ланцюгова блискавка» завдає цілі 10к8 ушкодже...
6,q07,парафраз,False,1,True,True,True,"Заклинання, яке виганяє істоту в деміплощину, ..."
7,q08,парафраз,False,1,True,True,True,"Заклинання, яке дозволяє підняти істоту вертик..."
8,q09,правила,False,1,True,True,True,Тривалий відпочинок триває щонайменше 8 годин ...
9,q10,правила,False,1,True,True,True,Короткий відпочинок триває мінімум в 1 годину ...


## 1.6. Аналіз відмов

Клітинка нижче збирає всі питання, на які система відповіла неправильно, і ставить **попередній** тип відмови за простими правилами. Остаточний тип визначаю вручну після прочитання відповіді й фрагментів: суддя — та сама Qwen 4B, тож його вердикт теж може бути помилковим.

In [ ]:
def failure_kind(r):
    if r["answerable"]:
        if not r["gold_in_context"]:
            return "промах ретриву"
        if r["refused"]:
            return "хибна відмова попри наявний контекст"
        if not r["judge"]:
            return "помилка генерації або судді (перевірити вручну)"
        if not r["cites_gold"]:
            return "правильна відповідь, але посилання не на той фрагмент"
        return None
    return None if (r["refused"] and r["n_cited"] == 0) else "галюцинація: відповідь на питання без відповіді в корпусі"

QBY = {q[0]: q for q in BENCH}
FAILS = [r for r in ROWS if failure_kind(r)]
print(f"проблемних питань: {len(FAILS)} з {len(ROWS)}")
for r in FAILS:
    q = QBY[r["id"]]
    gold = [i for i, c in enumerate(CH) if is_gold(q, c)] if q[3] else []
    print("=" * 90)
    print(f'{r["id"]} [{r["type"]}] {q[2]}')
    print("попередній тип:", failure_kind(r))
    print("еталон:", q[5])
    print("відповідь:", r["answer"][:400])
    print("контекст:", [f'{CH[i]["title"][:40]} ({pages(CH[i])})' for i in r["ctx_ids"]])
    print("золоті фрагменти:", gold, "| з них у контексті:", [i for i in gold if i in r["ctx_ids"]])

проблемних питань: 5 з 26
q04 [заклинання] Який рівень має заклинання «Бажання» і до якої школи магії належить?
попередній тип: хибна відмова попри наявний контекст
еталон: 9 рівень, виклик.
відповідь: Заклинання «Бажання» має 9 рівень [1]. У наданих фрагментах не вказано, до якої школи магії воно належить. У фрагментах [1] і [3] згадується, що бажання може дублювати ефекти будь-яких заклинань з рівнем не вище 8, але ніде не вказано, до якої школи магії воно належить. У фрагментах [4] і [5] розглядаються інші заклинання та їх школи, але не «Бажання».  

У наданих фрагментах немає відповіді на це
контекст: ['Заклинання: БАЖАННЯ (с. 183)', 'РОЗДІЛ 10. ВИКОНАННЯ ЗАКЛИНАНЬ › ЩО ТАКЕ (с. 171)', 'РОЗДІЛ 3. КЛАСИ › СПРИТНИК (с. 84)', 'Заклинання: ЯСНІСТЬ РОЗУМУ (с. 256)', 'РОЗДІЛ 3. КЛАСИ › БОЄЦЬ (с. 40)']
золоті фрагменти: [759] | з них у контексті: [759]
q06 [заклинання] Яку шкоду завдає «Ланцюгова блискавка» і на скільки додаткових цілей вона перестрибує?
попередній тип: помилка генерації 

### Розбір випадків

Усі 5 питань, на які система відповіла неправильно за автоматичною перевіркою, розібрано вручну: прочитано відповідь, еталон і фрагменти, що потрапили в контекст. Остаточні типи для q04, q06 і q14 відрізняються від автоматичних попередніх.

---

#### 1. q04 [заклинання] Який рівень має заклинання «Бажання» і до якої школи магії належить? — Промах ретриву
* **Відповідь системи:** «У наданих фрагментах немає відповіді на це питання»
* **Причина:** Таблицю досвіду парсер «розплющив» у рядок чисел `0 1 +2 300 2 +2 …` без підписів і вклав в один фрагмент із прозою про категорії персонажів; назва фрагмента не містить слова «досвід». У топ-5 потрапили фрагменти про класи й мультикласування, де теж ідеться про рівні. Відмова моделі за такого контексту коректна, це не помилка генерації.
* **Що змінити:** Виділяти таблиці в окремі фрагменти з назвою; додавати підзаголовок розділу до назви фрагмента; збільшити пул кандидатів для reranker з 20 до 50.

---

#### 2. q06 [заклинання] Яку шкоду завдає «Ланцюгова блискавка» і на скільки додаткових цілей вона перестрибує? — Помилка судді
* **Відповідь системи:** «10к8 ушкоджень електрикою… перестрибує три розряди на інші цілі… в межах 6 метрів [1]»
* **Причина:** У фрагменті 893: *«три розряди перестрибують … на інші цілі, яких може бути не більше трьох»*. Відповідь називає шкоду й три розряди, але неточно формулює кількість цілей; суддя порівнював із еталоном «три додаткові цілі» й виніс НЕВІРНО. По суті відповідь майже правильна.
* **Що змінити:** Формулювати еталони короткими фактами; просити суддю перевіряти кожен факт окремо; звіряти вердикти з незалежною моделлю.

---

#### 3. q14 [правила] Що відбувається з осліпленою істотою під час атак? — Неоднозначний еталон
* **Відповідь системи:** «Атаки по осліпленій істоті — з перевагою, її атаки по інших — з перешкодою [1]»
* **Причина:** Відповідь точна щодо питання, але в еталоні було ще й автоматичне провалення перевірок, пов'язаних із зором, про що питання не питало. Суддя вимагав усіх фактів еталона.
* **Що змінити:** Звузити еталон до питання; у промпт судді додати «оцінюй лише те, про що запитано».

---

#### 4. q20 [спорядження] Який Клас Обладунку дає кольчуга і яка її ціна? — Галюцинація попри контекст і хибне цитування
* **Відповідь системи:** «КО 18, ціна 75 зм [2]»
* **Причина:** Правильний фрагмент був у контексті: *«Кольчуга 75 зм 16 Сил 13 … Лати 1,500 зм 18»*. Це розплющена таблиця: заголовок колонок лишився в попередньому фрагменті, а значення сусідніх рядків йдуть підряд без підписів, тож модель, імовірно, взяла КО сусіднього рядка. Посилання [2] веде на фрагмент розділу 1, який відповіді не містить.
* **Що змінити:** Парсити таблиці структуровано і записувати кожен рядок як *«Кольчуга: вартість 75 зм, КО 16, …»*; повторювати заголовок таблиці в кожному її фрагменті; перевіряти, що числа з відповіді є в процитованому фрагменті.


---

#### 5. q21 [таблиця] Скільки очок досвіду потрібно для 5 рівня? — Помилка генерації попри контекст
* **Відповідь системи:** «9 рівень [1]. У наданих фрагментах не вказано, до якої школи… У наданих фрагментах немає відповіді на це питання.»
* **Причина:** Фрагмент 759 починається з «9 рівень, виклик», тобто школа в контексті є, але без підпису «школа» модель 4B, імовірно, не зв'язала слово «виклик» зі школою магії. Відповідь поєднала правильну частину з хибним твердженням про відсутність даних і додала фразу відмови; `is_refusal` перевіряє підрядок, тому зарахував її як відмову.
* **Що змінити:** Записувати метадані явно; вважати відповідь відмовою лише якщо вона *починається* з фрази відмови; додати в промпт «якщо відповідь відома частково, відповідай на відому частину».

## 1.7. Демонстрація

Інтерфейс на Gradio: питання, відповідь і джерела (розділ, сторінка, текст фрагмента). Повзунок `k` змінює кількість фрагментів у контексті.

In [ ]:
!pip install -q gradio
import gradio as gr

def ui_answer(question, k):
    k = int(k)
    ids = retrieve(question, k)
    ctx = [CH[i] for i in ids]
    text, _ = chat(build_messages(question, ctx))
    sources = "\n\n".join(f"**[{n}] {c['title']}, {pages(c)}**\n\n{c['text'][:500]}…" for n, c in enumerate(ctx, start=1))
    return text, sources

demo = gr.Interface(
    fn=ui_answer,
    inputs=[gr.Textbox(label="Питання", lines=2), gr.Slider(1, 8, value=K, step=1, label="k (фрагментів у контексті)")],
    outputs=[gr.Textbox(label="Відповідь"), gr.Markdown(label="Джерела")],
    title="Книга Гравця D&D 5e: питання за правилами",
    examples=[["Яка дистанція дії заклинання «Вогняна куля»?", 5],
              ["Скільки триває тривалий відпочинок?", 5],
              ["Який показник небезпечності має беголдер?", 5]],
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f77ea49c35efa7f375.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Генеративна модель і результати відповідей

In [ ]:
from IPython.display import Markdown, display

S = summarize(ROWS)
avg_tok = sum(r["prompt_tokens"] for r in ROWS) / len(ROWS)
lines = [f"**Модель:** `{MODEL_ID}` (fp16, локально через `transformers`, `do_sample=False`). Контекст — k = {K} фрагментів "
         f"після гібридного пошуку з переранжуванням (конфігурація {BEST}), у середньому {avg_tok:.0f} токенів у промпті.", "",
         "| Метрика | Значення |", "|---|---|"]
lines += [f"| {k} | {v:.2f} |" for k, v in S.items()]
lines += ["", "| Тип питання | К-сть | Правильних (суддя) | Цитують золотий фрагмент |", "|---|---|---|---|"]
by_type = {}
for r in ROWS:
    by_type.setdefault(r["type"], []).append(r)
for t, rs in by_type.items():
    if rs[0]["answerable"]:
        lines.append(f"| {t} | {len(rs)} | {sum(r['judge'] for r in rs)} | {sum(r['cites_gold'] for r in rs)} |")
    else:
        lines.append(f"| {t} | {len(rs)} | коректних відмов: {sum(r['refused'] and r['n_cited'] == 0 for r in rs)} | — |")
display(Markdown("\n".join(lines)))

**Модель:** `Qwen/Qwen3-4B-Instruct-2507` (fp16, локально через `transformers`, `do_sample=False`). Контекст — k = 5 фрагментів після гібридного пошуку з переранжуванням (конфігурація 1200/150), у середньому 3296 токенів у промпті.

| Метрика | Значення |
|---|---|
| відповідь правильна (суддя), питання з відповіддю | 0.77 |
| золотий фрагмент у контексті (hit@k) | 0.95 |
| правильна І цитує золотий фрагмент | 0.77 |
| є хоч одне посилання [n] | 0.95 |
| хибна відмова (відповідь у корпусі є) | 0.09 |
| коректна відмова, питання без відповіді | 1.00 |

| Тип питання | К-сть | Правильних (суддя) | Цитують золотий фрагмент |
|---|---|---|---|
| заклинання | 6 | 4 | 6 |
| парафраз | 2 | 2 | 2 |
| правила | 6 | 5 | 6 |
| раса | 2 | 2 | 2 |
| клас | 2 | 2 | 2 |
| спорядження | 2 | 1 | 1 |
| таблиця | 1 | 0 | 0 |
| істота | 1 | 1 | 1 |
| без відповіді | 4 | коректних відмов: 4 | — |

### Аналіз відмов
Автоматична перевірка відзначила 5 проблемних питань з 26. Після ручного розбору (таблиця в 1.6) реальних помилок системи три, ще дві належать судді й еталону:

| Тип відмови | Питання | К-сть |
|---|---|---|
| промах ретриву | q21 | 1 |
| галюцинація попри контекст (разом із хибним цитуванням) | q20 | 1 |
| помилка генерації попри контекст (змішана відповідь і відмова) | q04 | 1 |
| помилка судді / неоднозначний еталон | q06, q14 | 2 |

Спільна причина двох із трьох реальних помилок (q20, q21) — втрата структури таблиць при парсингу: таблиця стає рядком чисел без заголовків колонок, а розріз на фрагменти може відірвати її від заголовка. Це найкритичніша проблема, бо вона дає не лише відмову, а й впевнено неправильну відповідь із хибним посиланням (q20). Ретрив не є вузьким місцем: золотий фрагмент потрапив у контекст у 21 з 22 питань (0.95), і на всі чотири питання без відповіді система відмовилась без посилань, зокрема на питання про беголдера, де в книзі є схожі стат-блоки інших істот.

## 1.6. Висновок
- **Ретрив.** Найкраща конфігурація — `1200/150` + гібридний пошук + переранжування. Семантичний пошук кращий за BM25, а переранжування дає найбільший приріст саме на hit@1.
- **Генерація.** Qwen3-4B-Instruct-2507 із k = 5 правильно відповіла на 17 з 22 питань за вердиктом судді; після ручного виправлення двох помилкових вердиктів q06, q14 — на 19 з 22. На 20 з 22 питань вона посилається на правильний фрагмент.
- **Відмова.** Система не вигадує відповіді там, де їх немає, ціною двох хибних відмов на питаннях з відповіддю: q21 через промах ретриву й q04 через змішану відповідь.
- **Обмеження.** Бенчмарк із 22 питань з відповіддю не розрізняє близькі конфігурації. Суддя — та сама модель, що генерує; вердикти «НЕВІРНО» перевірено вручну, а вердикти «ВІРНО» окремо не перевірялися, тому оцінка правильності може бути завищеною. Таблиці й стат-блоки втрачають структуру, списки заклинань за класами не індексуються.
- **Що покращити в першу чергу.** Структурований парсинг таблиць і повтор їхніх заголовків у фрагментах; явні підписи метаданих у фрагментах заклинань; визначення відмови за початком відповіді; вужчі еталони й незалежний суддя; більший пул кандидатів для reranker.